<a href="https://colab.research.google.com/github/vaishnavye/A.V.E.R.A/blob/main/avera_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install streamlit pandas numpy matplotlib seaborn jinja2 google-generativeai
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import time
from datetime import date
from jinja2 import Template
from google import genai
from google.genai import types

# -----------------------------------------------------------------------------
# 1. STREAMLIT CONFIGURATION & SETUP
# -----------------------------------------------------------------------------
st.set_page_config(
    page_title="AVERA — Actuarial Validation And Evaluation Risk Agent",
    page_icon="📊",
    layout="wide"
)

st.title("📊 AVERA")
st.subheader("Agentic Actuarial Data Quality Report Compiler")
st.markdown("---")

# Sidebar: API Key and Data Upload
st.sidebar.header("🔧 Configuration & Data")

# Try to automatically grab the key from your Colab Secrets baseline
try:
    from google.colab import userdata
    default_key = userdata.get("GEMINI_API_KEY")
except Exception:
    default_key = ""

# The sidebar box will now automatically fill up with your secret key!
api_key_input = st.sidebar.text_input(
    "Gemini API Key",
    value=default_key,
    type="password",
    help="Fetched automatically from Colab Secrets if available."
)

# Initialize Gemini Client using the active key
client = None
if api_key_input:
    try:
        client = genai.Client(api_key=api_key_input)
        st.sidebar.success("✅ Gemini Client Ready")
    except Exception as e:
        st.sidebar.error(f"Error initializing client: {e}")
else:
    st.sidebar.warning("🔑 Add your Gemini API Key to generate commentary.")

uploaded_file = st.sidebar.file_uploader("Upload CAS Dataset (e.g., ppauto_pos.csv)", type=["csv"])
# -----------------------------------------------------------------------------
# 2. CORE ACTUARIAL HELPER FUNCTIONS (From reserving&claims.ipynb)
# -----------------------------------------------------------------------------
def build_triangle(df_subset, field: str, mask_future: bool = True) -> pd.DataFrame:
    if field not in df_subset.columns:
        raise KeyError(f"Column '{field}' not found.")

    df_clean = df_subset.dropna(subset=['AccidentYear', 'DevelopmentLag'])
    df_agg = df_clean.groupby(['AccidentYear', 'DevelopmentLag'])[field].sum().reset_index()

    tri = (df_agg
           .pivot(index="AccidentYear", columns="DevelopmentLag", values=field)
           .sort_index())

    if mask_future:
        eval_year = int(tri.index.max())
        for ay in tri.index:
            for dev in tri.columns:
                if int(ay) + int(dev) - 1 > eval_year:
                    tri.loc[ay, dev] = np.nan
    return tri

# -----------------------------------------------------------------------------
# 3. CHECKER TOOLS (From reserving&claims.ipynb)
# -----------------------------------------------------------------------------
def completeness_checker(df_subset) -> dict:
    tri = build_triangle(df_subset, 'CumPaidLoss')
    eval_year = int(tri.index.max())
    missing = []
    for ay in tri.index:
        for dev in tri.columns:
            if int(ay) + int(dev) - 1 <= eval_year and pd.isna(tri.loc[ay, dev]):
                missing.append([int(ay), int(dev)])
    status = 'Green' if not missing else 'Red'
    return {
        'status': status,
        'missing_count': len(missing),
        'missing_cells': missing,
        'summary': f"{len(missing)} missing cell(s) in the known triangle.",
    }

def sign_checker(df_subset) -> dict:
    cum_tri = build_triangle(df_subset, 'CumPaidLoss')
    inc_tri = cum_tri.diff(axis=1)
    if not cum_tri.empty:
        inc_tri[cum_tri.columns[0]] = cum_tri[cum_tri.columns[0]]
    negatives = []
    for ay in inc_tri.index:
        for dev in inc_tri.columns:
            val = inc_tri.loc[ay, dev]
            if pd.notna(val) and val < 0:
                negatives.append([int(ay), int(dev), round(float(val), 2)])
    status = 'Green' if not negatives else 'Amber'
    return {
        'status': status,
        'negative_count': len(negatives),
        'negative_increments': negatives,
        'summary': f"{len(negatives)} negative incremental paid-loss cell(s).",
    }

def consistency_checker(df_subset) -> dict:
    paid_tri = build_triangle(df_subset, 'CumPaidLoss')
    incurr_tri = build_triangle(df_subset, 'IncurredLosses')
    violations = []
    for ay in paid_tri.index:
        for dev in paid_tri.columns:
            p = paid_tri.loc[ay, dev]
            i = incurr_tri.loc[ay, dev]
            if pd.notna(p) and pd.notna(i) and p > i + 1e-6:
                violations.append([int(ay), int(dev), round(float(p), 2), round(float(i), 2)])
    status = 'Green' if not violations else 'Red'
    return {
        'status': status,
        'violation_count': len(violations),
        'violations': violations,
        'summary': f"{len(violations)} cell(s) where Paid > Incurred.",
    }

def outlier_detector(raw_df, grcode, df_subset, z_threshold: float = 2.0) -> dict:
    col = 'CumPaidLoss'
    all_tri = raw_df.copy()
    eval_yr = int(raw_df['AccidentYear'].max())
    devs = sorted(raw_df['DevelopmentLag'].unique())

    industry_ldfs = {}
    for i in range(len(devs) - 1):
        d_from, d_to = devs[i], devs[i+1]
        sub_from = all_tri[all_tri['DevelopmentLag'] == d_from][['GRCODE', 'AccidentYear', col]].rename(columns={col: 'from_val'})
        sub_to = all_tri[all_tri['DevelopmentLag'] == d_to][['GRCODE', 'AccidentYear', col]].rename(columns={col: 'to_val'})
        merged = sub_from.merge(sub_to, on=['GRCODE', 'AccidentYear'])
        merged = merged[merged['AccidentYear'] + d_to - 1 <= eval_yr]
        merged = merged[(merged['from_val'] > 0) & (merged['to_val'] > 0)]
        if len(merged) >= 3:
            industry_ldfs[d_from] = merged['to_val'].values / merged['from_val'].values

    comp_tri = build_triangle(df_subset, 'CumPaidLoss')
    outliers = []
    for i in range(len(devs) - 1):
        d_from, d_to = devs[i], devs[i+1]
        if d_from not in industry_ldfs:
            continue
        ind_vals = industry_ldfs[d_from]
        mu, sigma = ind_vals.mean(), ind_vals.std()
        if sigma < 1e-9:
            continue
        for ay in comp_tri.index:
            num = comp_tri.loc[ay, d_to] if d_to in comp_tri.columns else np.nan
            den = comp_tri.loc[ay, d_from] if d_from in comp_tri.columns else np.nan
            if pd.notna(num) and pd.notna(den) and den > 0:
                ldf = num / den
                z = (ldf - mu) / sigma
                if abs(z) > z_threshold:
                    outliers.append([int(ay), int(d_from), round(ldf, 4), round(mu, 4), round(z, 2)])

    status = 'Green' if not outliers else 'Amber'
    return {
        'status': status,
        'outlier_count': len(outliers),
        'outlier_factors': outliers,
        'summary': f"{len(outliers)} LDF outlier(s) found vs cross-company benchmark (|z|>{z_threshold}).",
    }

# -----------------------------------------------------------------------------
# 4. LLM GENERATION WRAPPERS
# -----------------------------------------------------------------------------
def run_commentary_writer(client, check_name: str, check_result: dict) -> str:
    if not client:
        return "⚠️ Commentary generation skipped (Missing Gemini API Key)."

    prompt = f"""You are an actuarial data-quality assistant writing commentary for
a Data Quality Report section. Write 2-4 sentences in professional actuarial language
explaining the following finding to a reserving actuary.

Check: {check_name}
Result: {json.dumps(check_result, indent=2)}

Rules:
- Do NOT just repeat the numbers. Explain what they mean actuarially.
- Mention why the finding matters (or doesn't) for reserving / IBNR estimation.
- If status is Green say so clearly and briefly.
- If Amber or Red, explain the risk and suggest one specific investigation step.
- Keep it under 80 words."""

    try:
        resp = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config=types.GenerateContentConfig(temperature=0.3, max_output_tokens=200)
        )
        return resp.text.strip()
    except Exception as e:
        return f"Error generating commentary: {e}"

def generate_next_steps(client, all_summaries: str) -> list:
    if not client:
        return ["Add API Key to generate professional next steps."]

    ns_prompt = f"""Based on these data quality findings for a PP Auto loss triangle:
{all_summaries}
List exactly 4 concise, actionable next steps for the reserving actuary.
Format as a plain numbered list. No headers. Each step under 20 words."""

    try:
        ns_resp = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=ns_prompt,
            config=types.GenerateContentConfig(temperature=0.2, max_output_tokens=200)
        )
        steps = [line.strip().lstrip('1234567890. ').strip()
                 for line in ns_resp.text.strip().split('\n') if line.strip()]
        return steps[:4]
    except Exception as e:
        return [f"Error generating next steps: {e}"]

# -----------------------------------------------------------------------------
# 5. JINJA REPORT COMPILER TEMPLATE (From reserving&claims.ipynb)
# -----------------------------------------------------------------------------
REPORT_TEMPLATE = """# AVERA — Data Quality Report

| Field | Value |
|---|---|
| **Company** | {{ company_name }} (GRCODE {{ grcode }}) |
| **Line of Business** | Private Passenger Auto Liability/Medical (Schedule P) |
| **Report Date** | {{ report_date }} |
| **Overall Readiness** | {{ overall_verdict }} |

---

## Overall Verdict: {{ overall_verdict }}
{{ overall_rationale }}

---

## Findings by Check

{% for section in sections %}
### {{ loop.index }}. {{ section.title }} — {{ section.status }}

**Raw findings:** {{ section.raw_summary }}

**Actuarial commentary:** {{ section.commentary }}

{% endfor %}

---

## Recommended Next Steps
{% for step in next_steps %}
{{ loop.index }}. {{ step }}
{% endfor %}

---
*This report was generated by AVERA (Gemini 2.5 Flash agentic system). All findings are flagged for human review — the final decision to proceed with analysis rests with the responsible actuary.*"""

# -----------------------------------------------------------------------------
# MAIN APP FLOW
# -----------------------------------------------------------------------------
if uploaded_file is not None:
    raw_df = pd.read_csv(uploaded_file)

    required_cols = ['GRCODE', 'GRNAME', 'AccidentYear', 'DevelopmentLag', 'CumPaidLoss', 'IncurredLosses', 'EarnedPremNet']
    if not all(col in raw_df.columns for col in required_cols):
        st.error(f"The dataset lacks some required columns. Make sure it contains: {required_cols}")
        st.stop()

    tab1, tab2 = st.tabs(["🎯 Single Company Audit Pipeline", "🏢 Multi-Company Cross Comparison"])

    with tab1:
        company_list = raw_df['GRCODE'].unique()
        selected_grcode = st.selectbox("Select Company NAIC Code (GRCODE):", options=company_list,
                                       format_func=lambda x: f"[{x}] {raw_df[raw_df['GRCODE']==x]['GRNAME'].iloc[0]}")

        company_df = raw_df[raw_df['GRCODE'] == selected_grcode].copy()
        company_df['AccidentYear'] = pd.to_numeric(company_df['AccidentYear'], errors='coerce')
        company_df['DevelopmentLag'] = pd.to_numeric(company_df['DevelopmentLag'], errors='coerce')
        company_name = company_df['GRNAME'].iloc[0]

        prem = company_df.groupby('AccidentYear')['EarnedPremNet'].max()

        col1, col2, col3 = st.columns(3)
        col1.metric("Rows Evaluated", f"{len(company_df):,}")
        col2.metric("Total Net Earned Premium", f"${round(float(prem.sum()), 0):,}")
        col3.metric("Avg Premium / Year", f"${round(float(prem.mean()), 0):,}")

        if st.button("🔥 Run Audit & Compile Report", key="run_single"):
            with st.spinner("Processing actuarial calculations and generating AI summaries..."):

                comp_res = completeness_checker(company_df)
                sign_res = sign_checker(company_df)
                cons_res = consistency_checker(company_df)
                out_res = outlier_detector(raw_df, selected_grcode, company_df)

                comm_comp = run_commentary_writer(client, 'completeness_checker', comp_res)
                comm_sign = run_commentary_writer(client, 'sign_checker', sign_res)
                comm_cons = run_commentary_writer(client, 'consistency_checker', cons_res)
                comm_out = run_commentary_writer(client, 'outlier_detector', out_res)

                all_summaries = f"{comp_res['summary']}\n{sign_res['summary']}\n{cons_res['summary']}\n{out_res['summary']}"
                next_steps_list = generate_next_steps(client, all_summaries)

                statuses = [comp_res['status'], sign_res['status'], cons_res['status'], out_res['status']]
                if 'Red' in statuses:
                    overall = '🔴 RED — Data Not Ready'
                elif 'Amber' in statuses:
                    overall = '🟡 AMBER — Proceed With Caution'
                else:
                    overall = '🟢 GREEN — Data Ready'

                overall_rationale = (f'Out of 4 checks, {statuses.count("Red")} returned Red, '
                                     f'{statuses.count("Amber")} returned Amber, and {statuses.count("Green")} returned Green.')

                sections = [
                    {'title': 'Completeness Check', 'status': comp_res['status'], 'raw_summary': comp_res['summary'], 'commentary': comm_comp},
                    {'title': 'Sign Check', 'status': sign_res['status'], 'raw_summary': sign_res['summary'], 'commentary': comm_sign},
                    {'title': 'Paid vs Incurred Check', 'status': cons_res['status'], 'raw_summary': cons_res['summary'], 'commentary': comm_cons},
                    {'title': 'LDF Outlier Detection', 'status': out_res['status'], 'raw_summary': out_res['summary'], 'commentary': comm_out},
                ]

                report_md = Template(REPORT_TEMPLATE).render(
                    company_name=company_name,
                    grcode=selected_grcode,
                    report_date=date.today().isoformat(),
                    overall_verdict=overall,
                    overall_rationale=overall_rationale,
                    sections=sections,
                    next_steps=next_steps_list,
                )

                st.markdown("---")
                st.header("📋 Compiled Data Quality Report")
                st.markdown(report_md)

                fname = f'DQR_PPAuto_{selected_grcode}_{date.today().isoformat()}.md'
                st.download_button("📥 Download Report (.md)", data=report_md, file_name=fname, mime="text/markdown")

                st.markdown("---")
                st.header("📈 Appendix: Visualizations")

                v_col1, v_col2 = st.columns(2)

                with v_col1:
                    tri = build_triangle(company_df, 'CumPaidLoss')
                    fig1, ax1 = plt.subplots(figsize=(10, 6))
                    sns.heatmap(tri/1e6, mask=tri.isna(), annot=True, fmt='.1f', cmap='Blues',
                                linewidths=0.5, ax=ax1, cbar_kws={'label': 'Cum Paid Loss ($M)'})
                    ax1.set_title(f'Cumulative Paid Losses — PP Auto\n{company_name}', fontweight='bold')
                    ax1.set_xlabel('Development Lag (years)')
                    ax1.set_ylabel('Accident Year')
                    st.pyplot(fig1)

                with v_col2:
                    devs = sorted(tri.columns)
                    ldf_data, labels = [], []
                    for i in range(len(devs)-1):
                        d_from, d_to = devs[i], devs[i+1]
                        ratios = []
                        for ay in tri.index:
                            n, d = tri.loc[ay, d_to], tri.loc[ay, d_from]
                            if pd.notna(n) and pd.notna(d) and d > 0:
                                ratios.append(n/d)
                        if ratios:
                            ldf_data.append(ratios)
                            labels.append(f'{d_from}-{d_to}')

                    if ldf_data:
                        fig2, ax2 = plt.subplots(figsize=(10, 6))
                        ax2.boxplot(ldf_data, labels=labels, patch_artist=True, boxprops=dict(facecolor='#AED6F1'))
                        ax2.axhline(1.0, color='red', linestyle='--', alpha=0.5, label='LDF = 1.0')
                        ax2.set_title(f'LDF Distribution per Period\n{company_name}', fontweight='bold')
                        ax2.set_xlabel('Development Period')
                        ax2.set_ylabel('Age-to-Age LDF')
                        ax2.legend()
                        st.pyplot(fig2)

                fig3, ax3 = plt.subplots(figsize=(10, 3))
                ax3.bar(prem.index, prem.values/1e6, color='#2E86C1', alpha=0.8)
                ax3.set_title(f'Net Earned Premium by Accident Year — {company_name}', fontweight='bold')
                ax3.set_xlabel('Accident Year')
                ax3.set_ylabel('Earned Premium ($M)')
                st.pyplot(fig3)

    with tab2:
        st.header("🏢 Multi-Company Readiness Benchmarking")
        st.markdown("Run structural framework evaluation metrics on multiple groups simultaneously.")

        n_companies = st.slider("Number of Companies to Scan", min_value=2, max_value=20, value=5)

        if st.button("📊 Generate Cross-Comparison Matrix", key="run_multi"):
            sample_codes = raw_df['GRCODE'].unique()[:n_companies]
            summary_rows = []

            progress_bar = st.progress(0)
            for idx, gc in enumerate(sample_codes):
                sub_df = raw_df[raw_df['GRCODE'] == gc].copy()

                comp_r = completeness_checker(sub_df)
                sign_r = sign_checker(sub_df)
                cons_r = consistency_checker(sub_df)
                out_r = outlier_detector(raw_df, int(gc), sub_df)

                statuses = [comp_r['status'], sign_r['status'], cons_r['status'], out_r['status']]
                if 'Red' in statuses:
                    verdict = '🔴 RED'
                elif 'Amber' in statuses:
                    verdict = '🟡 AMBER'
                else:
                    verdict = '🟢 GREEN'

                gname = raw_df[raw_df['GRCODE'] == gc]['GRNAME'].iloc[0]
                summary_rows.append({
                    'GRCODE': gc,
                    'Company Name': gname[:35],
                    'Completeness': "🟢" if comp_r['status'] == 'Green' else "🔴",
                    'Sign Check': "🟢" if sign_r['status'] == 'Green' else "🟡",
                    'Paid ≤ Incurred': "🟢" if cons_r['status'] == 'Green' else "🔴",
                    'LDF Outliers': "🟢" if out_r['status'] == 'Green' else "🟡",
                    'Overall Verdict': verdict,
                })
                progress_bar.progress((idx + 1) / len(sample_codes))
                time.sleep(0.1)

            summary_df = pd.DataFrame(summary_rows)
            st.dataframe(summary_df, use_container_width=True)

else:
    st.info("👋 Welcome! Please upload a loss development dataset (`.csv` file) using the sidebar menu to get started.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 51.9 MB/s eta 0:00:00


2026-06-28 02:10:21.392 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-28 02:10:21.394 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-28 02:10:21.588 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-06-28 02:10:21.589 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-28 02:10:21.590 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-28 02:10:21.591 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-28 02:10:21.592 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

In [ ]:
# 1. Install necessary dependencies and the tunnel tool
!pip install -q streamlit google-genai pandas numpy matplotlib seaborn jinja2
!npm install -g localtunnel

# 2. Launch your avera.py script in the background
import subprocess
subprocess.Popen(["streamlit", "run", "gemini3.py", "--server.port", "8501"])

# 3. Retrieve your unique Tunnel Password (IP Address)
print("\n🔑 STEP 1: COPY THIS PASSWORD/IP ADDRESS EXACTLY:")
!curl ipv4.icanhazip.com

# 4. Generate the public access web link
print("\n🌐 STEP 2: CLICK THE LINK BELOW TO OPEN AVERA:")
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏
changed 22 packages in 2s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏
🔑 STEP 1: COPY THIS PASSWORD/IP ADDRESS EXACTLY:
34.186.38.140

🌐 STEP 2: CLICK THE LINK BELOW TO OPEN AVERA:
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙your url is: https://floppy-cups-make.loca.lt
